# Build a Simple RAG System

#### Install OpenAI, and LangChain dependencies

In [1]:
!pip install langchain==0.3.10

  Using cached langchain-0.3.10-py3-none-any.whl.metadata (7.1 kB)
Using cached langchain-0.3.10-py3-none-any.whl (1.0 MB)
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.25
    Uninstalling langchain-0.3.25:
      Successfully uninstalled langchain-0.3.25


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.11 requires langchain<0.4.0,>=0.3.11, but you have langchain 0.3.10 which is incompatible.


In [2]:
!pip install langchain-openai==0.2.12

In [3]:
!pip install langchain-community==0.3.11

  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.78-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.77-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.76-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_core-0.3.75-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_core-0.3.74-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-0.3.73-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-0.3.72-py3-none-any.whl.metadata (5.8 kB)
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requir

In [5]:
!pip install dill

In [6]:
!pip install langchain-chroma==0.1.4

In [8]:
# Enter OpenAI API Key
from getpass import getpass
OPENAI_API_KEY = getpass("Enter your OpenAI API Key here: ")

In [9]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(OPENAI_API_KEY)

In [10]:

# Open AI Embedding Models
from langchain_openai import OpenAIEmbeddings
openai_embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Loading and Processing the Data

In [5]:
# if you can't download using the following code
# go to https://drive.google.com/file/d/1QkSY9W5RyaBnY8c5FLIsmpPVXoHTQ-fb/view?usp=sharing download it
# manually upload it on colab
!gdown 1QkSY9W5RyaBnY8c5FLIsmpPVXoHTQ-fb

Downloading...
From: https://drive.google.com/uc?id=1QkSY9W5RyaBnY8c5FLIsmpPVXoHTQ-fb
To: /content/rag_eval_docs.csv
100% 2.66k/2.66k [00:00<00:00, 13.8MB/s]


## Load and Process the dataset

In [11]:
import pandas as pd
df = pd.read_csv('29th Oct 2025-20251105T052934Z-1-001/29th Oct 2025/rag_eval_docs.csv')
df

,id,title,context
0,1,Machine Learning,Machine learning is a field of artificial inte...
1,2,Deep Learning,Deep learning is a subset of machine learning ...
2,3,Natural Language Processing (NLP),NLP is a branch of AI that enables computers t...
3,4,Pyramids,"Pyramids are ancient structures, often serving..."
4,5,Photosynthesis,Photosynthesis is the process plants use to co...
5,6,Biology,"Biology is the study of living organisms, cove..."
6,7,Quantum Mechanics,Quantum mechanics is a branch of physics that ...
7,8,Cryptocurrency,Cryptocurrency is a digital currency that uses...
8,9,Renewable Energy,"Renewable energy sources, such as solar and wi..."
9,10,Artificial Intelligence,Artificial intelligence refers to machines mim...


In [12]:
docs = df.to_dict(orient='records')
docs[:3]

[{'id': 1,
  'title': 'Machine Learning',
  'context': 'Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.'},
 {'id': 2,
  'title': 'Deep Learning',
  'context': 'Deep learning is a subset of machine learning utilizing neural networks with many layers. It excels in complex tasks like image and speech recognition. Convolutional and recurrent neural networks are among the common architectures used.'},
 {'id': 3,
  'title': 'Natural Language Processing (NLP)',
  'context': 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.'}]

In [13]:
from langchain.docstore.document import Document
processed_docs = []

for doc in docs:
  metadata = {
      "title": doc['title'],
      "id": doc['id']
  }
  data = doc['context']
  processed_docs.append(Document(page_content=data, metadata=metadata))
processed_docs[:3]

[Document(metadata={'title': 'Machine Learning', 'id': 1}, page_content='Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.'),
 Document(metadata={'title': 'Deep Learning', 'id': 2}, page_content='Deep learning is a subset of machine learning utilizing neural networks with many layers. It excels in complex tasks like image and speech recognition. Convolutional and recurrent neural networks are among the common architectures used.'),
 Document(metadata={'title': 'Natural Language Processing (NLP)', 'id': 3}, page_content='NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.')]

In [14]:
# Index Document Chunking and Embeddings in VectorDB (ChromaDB)

from langchain_chroma import Chroma
chroma_db = Chroma.from_documents(documents=processed_docs,
                                  collection_name='my_db',embedding=openai_embed_model,
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory='./my_db')


In [15]:
# Load Vector DB from disk
chroma_db = Chroma(persist_directory='./my_db', collection_name='my_db',
                   embedding_function=openai_embed_model)

# Semantic Similarity Based Retrieval

In [16]:
# Retrieval part
similarity_retriever = chroma_db.as_retriever(search_type='similarity_score_threshold',
                                              search_kwargs={"k": 3, "score_threshold":0.3})

In [17]:
from IPython.display import display, Markdown
def display_docs(docs):
  for doc in docs:
    print("Metadata: ", doc.metadata)
    print("Content Brief: ")
    display(Markdown(doc.page_content))
    print()

In [18]:
query = "What is AI?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata:  {'id': 10, 'title': 'Artificial Intelligence'}
Content Brief: 


Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.


Metadata:  {'id': 10, 'title': 'Artificial Intelligence'}
Content Brief: 


Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.


Metadata:  {'id': 3, 'title': 'Natural Language Processing (NLP)'}
Content Brief: 


NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.

In [22]:
query = "what does it need for plants to survive?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata:  {'id': 5, 'title': 'Photosynthesis'}
Content Brief: 


Photosynthesis is the process plants use to convert sunlight into energy. This process produces glucose and releases oxygen as a byproduct. It is crucial for sustaining life on Earth by providing food and oxygen.


Metadata:  {'id': 5, 'title': 'Photosynthesis'}
Content Brief: 


Photosynthesis is the process plants use to convert sunlight into energy. This process produces glucose and releases oxygen as a byproduct. It is crucial for sustaining life on Earth by providing food and oxygen.

# Build the RAG Pipeline

In [23]:
# Augmentation part
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """You are an assistant who is an expert in question-answering tasks.
              Answer the following question using only the following pieces of retrieved context.
              If the answer is not in the context, do not make up answers, just say that
              you don't know. Keep the answer to the point based on the information from the context.

              Question:
              {question}

              Context:
              {context}

              Answer:
              """
rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [24]:
# LLM model (Langchain model for building RAG System)
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

In [ ]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

src_rag_response_chain = (
    {
        "context": (itemgetter("context") | RunnableLambda(format_docs)),
        "question": RunnablePassthrough(),
    }
    | rag_prompt_template
    | chatgpt
    | StrOutputParser()
)


rag_chain_w_sources = (
    {"context": similarity_retriever, "question": RunnablePassthrough()}
    
    | RunnablePassthrough.assign(response = src_rag_response_chain))
    

In [26]:
query = "How do plants survive?"
result = rag_chain_w_sources.invoke(query)
result

{'context': [Document(metadata={'id': 5, 'title': 'Photosynthesis'}, page_content='Photosynthesis is the process plants use to convert sunlight into energy. This process produces glucose and releases oxygen as a byproduct. It is crucial for sustaining life on Earth by providing food and oxygen.'),
  Document(metadata={'id': 5, 'title': 'Photosynthesis'}, page_content='Photosynthesis is the process plants use to convert sunlight into energy. This process produces glucose and releases oxygen as a byproduct. It is crucial for sustaining life on Earth by providing food and oxygen.')],
 'question': 'How do plants survive?',
 'response': 'Plants survive by using photosynthesis to convert sunlight into energy, producing glucose for food and releasing oxygen as a byproduct.'}

In [27]:
query = "What is AI?"
result = rag_chain_w_sources.invoke(query)
result

{'context': [Document(metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(metadata={'id': 3, 'title': 'Natural Language Processing (NLP)'}, page_content='NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications rang

# Create a Synthetic RAG Golden Reference Dataset

In [28]:
doc_contexts = [doc.page_content for doc in processed_docs]
doc_contexts[:3]

['Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.',
 'Deep learning is a subset of machine learning utilizing neural networks with many layers. It excels in complex tasks like image and speech recognition. Convolutional and recurrent neural networks are among the common architectures used.',
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [34]:
!pip install deepeval

In [33]:
from deepeval.synthesizer import Synthesizer

synthesizer = Synthesizer(model ="gpt-4o")
eval_data = synthesizer.generate_goldens(
    # Provide a list of context for synthetic data generation
    documents = [[doc] for doc in doc_contexts],
    include_expected_output=True,
    max_goldens_per_context=1,
    num_evolutions=1,
    scenario="Retrieval Augmented Generation",
    task="Question Answering",
    evolutions = {
        "reasoning": 0.1,
        "multicontext": 0.9,
        "concretizing": 0.9,
        "constrained": 0.0,
        "comparative":0.0,
        "hypothetical": 0.0,
        "in_breadth": 0.0,
    }
)
    

AttributeError: 'Synthesizer' object has no attribute 'generate_goldens'

KeyboardInterrupt: 

In [36]:
query = "What is AI?"
response = rag_chain_w_sources.invoke(query)
response

{'context': [Document(metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(metadata={'id': 10, 'title': 'Artificial Intelligence'}, page_content="Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning."),
  Document(metadata={'id': 3, 'title': 'Natural Language Processing (NLP)'}, page_content='NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications rang

In [37]:
retrieved_context = [doc.page_content for doc in response['context']]
retrieved_context

["Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [38]:
human_answer = """AI, also known as Artificial Intelligence is used to build complex systems for applications
              like virtual assistants, robotics and autonomous vehicles."""

In [39]:
new_context = ["Machine Learning is the study of algorithms which learn with more data",
               "AI is known for Artificial Intelligence"] + retrieved_context
new_context

['Machine Learning is the study of algorithms which learn with more data',
 'AI is known for Artificial Intelligence',
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.",
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [40]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric
from deepeval import evaluate


test_case = LLMTestCase(
    input = response['question'],
    actual_output= response['response'],
    expected_output=human_answer,
    retrieval_context=new_context
)
test_case1 = LLMTestCase(
    input = response['question'],
    actual_output= response['response'],
    expected_output=human_answer,
    retrieval_context=retrieved_context
)


metrics = ContextualPrecisionMetric(
    threshold=0.50,
    model ="gpt-4o",
    include_reason=True,
    verbose_mode=True
)
result = evaluate([test_case1], [metrics])

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4o, strict=False, async_mode=True)...

Output()

Verdicts:
[
    {
        "verdict": "yes",
        "reason": "The context explains that 'Artificial intelligence refers to machines mimicking human 
intelligence' and includes applications like 'virtual assistants, robotics, and autonomous vehicles,' which aligns 
with the expected output."
    },
    {
        "verdict": "yes",
        "reason": "This context is identical to the first, providing the same relevant information about AI and its
applications, which supports the expected output."
    },
    {
        "verdict": "no",
        "reason": "The context focuses on 'NLP as a branch of AI' and its specific applications, which does not 
directly address the broader question of 'What is AI?' as required by the expected output."
    }
]
 
Score: 1.0
Reason: The score is 1.00 because the relevant nodes are perfectly ranked above the irrelevant node. The first node
provides a comprehensive explanation that 'Artificial intelligence refers to machines mimicking human intelligence'
and includes applications like 'virtual assistants, robotics, and autonomous vehicles,' which aligns with the 
expected output. The second node reinforces this by offering identical relevant information. The third node, which 
focuses on 'NLP as a branch of AI,' is correctly ranked lower as it does not directly address the broader question 
of 'What is AI?' as required by the expected output. Great job on maintaining perfect precision!

======================================================================



Metrics Summary

  - ✅ Contextual Precision (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o, reason: The score is 1.00 because the relevant nodes are perfectly ranked above the irrelevant node. The first node provides a comprehensive explanation that 'Artificial intelligence refers to machines mimicking human intelligence' and includes applications like 'virtual assistants, robotics, and autonomous vehicles,' which aligns with the expected output. The second node reinforces this by offering identical relevant information. The third node, which focuses on 'NLP as a branch of AI,' is correctly ranked lower as it does not directly address the broader question of 'What is AI?' as required by the expected output. Great job on maintaining perfect precision!, error: None)

For test case:

  - input: What is AI?
  - actual output: Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virt

⚠ WARNING: No hyperparameters logged.
» ]8;id=176936;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.44s | token cost: 0.005522500000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.